In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install google-api-core

In [3]:
!python --version
!curl ipecho.net/plain

Python 3.11.13
34.23.242.180

In [4]:
import os
import json
import pandas as pd
import datetime
import time
import random
import google.generativeai as genai
from google.api_core.exceptions import TooManyRequests, InternalServerError
import re


### API and model initialization

In [5]:
# Retrieve API key via Colab's secrets
from google.colab import userdata
api_key = userdata.get('GOOGLE_API_KEY')

In [6]:
# Change working directory and load progress

%cd '/content/drive/MyDrive/Colab Notebooks/DS310_NLP/annotation'
file_input = "comment_25.03.16.csv"
output_path = "docs/gemma"
key_index, token_usage, text_idx, text_id = 0, 0, 0, 0
try:
    with open('docs/gemma.json', 'r') as json_file:
        progress_data = json.load(json_file) # output type: dictionary

    token_usage = progress_data["token_usage"] if "token_usage" in progress_data else 0
    request_usage = progress_data["request_usage"] if "request_usage" in progress_data else 0
    text_idx = progress_data["text_idx"] if "text_idx" in progress_data else 0
    text_id = progress_data["text_id"] if "text_id" in progress_data else None

except Exception as e:
    print("Error:", e)

oken_usage = 0 if token_usage is None or token_usage < 0 else token_usage
text_idx = 0 if text_idx is None or text_idx < 0 else text_idx

print(key_index, token_usage, request_usage, text_idx, text_id, sep='\t\t')
start_point = 0 if text_idx == 0 else text_idx + 1
print(key_index, '\s', '\s', start_point, sep='\t\t')

/content/drive/MyDrive/Colab Notebooks/DS310_NLP/annotation
0		1979		2		12065		tik010300
0		\s		\s		12066


In [7]:
# Initialize client with the 1st key
genai.configure(api_key=api_key)

In [8]:

generation_config = {
    "temperature": 0.5, # Control randomness model's output. Tăng để tăng tính sáng tạo và đa dạng, khuyến khích tạo a nhiều nhãn hơn. 0.7 ổn hơn 0.8 theo GoEmotions.
    "top_p": 0.9, # control randomness using nucleus samling. value 1 (các tokens được chọn cộng lại==1). Giảm chút để tập trung vào các token có khả năng cao hơn, nưng vẫn cho phép 1 số tính ngẫu nhiên.
    "top_k": 27, # Tăng để xem xét nhiều lựa chọn token hơn, tạo cơ hội cho nhiều nhãn hơn được chọn. Giới hạn số lượng token được xem xét tương ứng với số lượng lớp.
    "max_output_tokens": 5000, # Không gian tạo ra nhãn và giải thích
}

# list of harm category dictionaries. BLOCK_NONE: not block response but still remain warning. Default: Block most
safety_settings = [
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
]

In [9]:
# models = genai.list_models()

# for model in models:
#   if 'generateContent' in model.supported_generation_methods:
#     print(model.name)

In [10]:
# Initialize GenerativeModel
def config_model():
    model = genai.GenerativeModel(
        model_name="gemma-3-27b-it", generation_config=generation_config, \
        safety_settings=safety_settings
    )

    return model

model =  config_model()
# rate limit: https://ai.google.dev/gemini-api/docs/rate-limits#free-tier
RPM = 30
TPM = 15_000
RPD = 14_400
tokens = 0
requests = 0

In [11]:
prompts = []
with open('docs/llm_guideline.md', 'r', encoding='utf-8') as f:
    content = f.read()
    sections = content.split('\n## ') # Split the content into sections based on ""
    sections = [sections[0]] + ['## ' + section for section in sections[1:]]
    prompts = [sections[0]]

display(prompts[0])

'## Prompt for multi-labels annotation\n\n**Objective**: Classify Vietnamese social media comment into one or more of the 27 emotion categories defined by GoEmotions, plus neutral.\n\n---\n\n### **Label Categories**\n- **amusement** (Giải trí) 😂: Cảm giác buồn cười trước nội dung hài hước, thú vị.\n- **excitement** (Hào hứng) 🤩: Cảm giác vui mừng, phấn khích trước sự kiện, thông tin tích cực.\n- **joy** (Niềm vui) 😀: Cảm giác hạnh phúc, vui vẻ, thoải mái trước tình huống tích cực.\n- **love** (Tình yêu) ❤️: Bày tỏ yêu thương, gắn bó với người, vật, hoặc ý tưởng.\n- **desire** (Mong muốn)\xa0😍: Cảm giác mạnh mẽ về mong muốn có được ai đó hoặc điều gì đó.\n- **optimism** (Lạc quan) 🤞: Hy vọng, tin tưởng vào kết quả tốt đẹp cho bản thân hoặc đối tượng khác.\n- **caring** (Quan tâm) 🤗: Thể hiện tình cảm, sự chăm sóc, hoặc hỗ trợ người khác.\n- **pride** (Tự hào) 😌: Hài lòng, kiêu hãnh về bản thân hoặc người khác.\n- **admiration** (Ngưỡng mộ) 👏: Kính trọng, khâm phục ai đó vì phẩm chất hoặ

In [12]:
# ******* IMPORTANT: You'll need to implement this based on your safety model *********
def should_process_tweet(tweet):
    """
    This function is where you would integrate a separate safety model or API.
    For now, it includes a basic keyword check as a placeholder.
    """
    # Basic keyword example (replace with your safety model's logic)
    # block_words = ["hate", "violence", "kill"]  # Extremely simplified!
    # if any(word in tweet.lower() for word in block_words):
    #     return False


    return True  # Default to allowing the tweet if no safety concerns found
# ***********************************************************************************


In [13]:

# Rate limiting parameters and mutable variables
REQUEST_INTERVAL = 60 / RPM # seconds per request
key_status = {}
max_retries = 5 # gemini sometimes has errors not relating to API
# immutable
last_request_time = 0 # initialize  the last request timestamp
retries = 0

# Retry function with exponential backoff
def retry_with_backoff(api_call, backoff_factor=2, max_wait=60):
    global last_request_time, key_index, model, retries # declare as global
    while retries < max_retries:
        # rate limiting: check if enough time has passed since the last request
        time_since_last_request = time.time() - last_request_time
        if time_since_last_request < REQUEST_INTERVAL:
            sleep_duration = REQUEST_INTERVAL - time_since_last_request
            print(f"Rate limiting: Sleeping for {sleep_duration:.2f} seconds...")
            time.sleep(sleep_duration)

        try:
            response = api_call()
            last_request_time = time.time() # Update the last request timestamp
            return response

        except Exception as e:
            wait_time = min(backoff_factor ** retries + random.uniform(0, 1), max_wait)
            print(f"Retrying in {wait_time:.2f} seconds...")
            time.sleep(wait_time)
            retries += 1
    raise Exception("Max retries exceeded")


## Massive annotating

In [14]:
stop_point = -1 # default: set any value < 0
print(f'start at index: {start_point}, stop at index: {stop_point}')

start at index: 12066, stop at index: -1


In [15]:
# Check start and stop points
df = pd.read_csv(file_input)
if stop_point < 0:
    stop_point = len(df) - 1

start_row = df.loc[start_point]
print("Start at index:", start_point)
print(f"Start at, id: {start_row['id'], start_row['text']}")
stop_row = df.loc[stop_point]
print("Stop at index:", stop_point)
print(f"Stop at id: {stop_row['id'], stop_row['text']}")

Start at index: 12066
Start at, id: ('tik010301', 'Ơ e xl nhma.e xem video th nhma e sợ bà của a mất thương bà quá để dành từng đồng cho con cháu')
Stop at index: 25959
Stop at id: ('thr000363', 'Bồ đã dốc lòng yêu một người rồi thì việc chia tay sẽ khiến bồ suy sụp, buồn dữ lắm nhưng mà cô gái oii dù có buồn cỡ nào, đau cỡ nào cũng không được năn nỉ hay van xin họ bất cứ điều gì cả. Nếu đủ yêu, đủ kiên nhẫn, đủ quyết tâm thì họ đã không quyết định buông tay rồi bồ ạ. Cứ buồn, cứ khóc, chấp nhận việc mình và họ đã không còn là gì của nhau nữa rồi, xem lại những tấm hình cũ, đến những nơi có nhiều kỉ niệm. Đừng cầu xin tình cảm từ một người đã muốn rời đi bồ nhé. Chúc bồ sớm move on!!!')


In [16]:
def clean_text(text):
    text = re.sub(r'\s+', ' ',  text)
    text = text.replace('.\n', '. ').replace('. \n', '. ').replace('\n', '. ').replace('.<br>', '. ').replace('<br>', '. ')
    # Replace specific HTML entities
    text = text.replace('&gt;', '>').replace('&lt;', '<').replace('&amp;', '&').replace('&quot;', '"').replace('&#39;', "'").replace('&nbsp;', ' ')
    return text


def generate_response(contents=None):
    """
    Annotation by LLM
    Args:
        contents: request content
    Returns:
        response: LLM response
    """
    try:
        response = retry_with_backoff(lambda: model.generate_content(contents=contents), backoff_factor=3, max_wait=180)
    except Exception as e:
        print(f"Stop due to error:", e)
        return -1

    return response


def save_progress(row:pd.core.series.Series):
    progress_data = {
        "date_time": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "key_index": key_index,
        "key": api_key,
        "token_usage": tokens,
        "request_usage": requests,
        "text_idx": row.name,
        "text_id": row['id']
    }

    # Save progress to JSON file
    with open('docs/gemma.json', 'w') as json_file:
        json.dump(progress_data, json_file)

    # Append progress to log file
    with open('docs/gemma_log.txt', 'a') as log_file:  # 'a' for append mode
        log_entry = f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}, key_index: {key_index}, key: {api_key}, token_usage: {tokens}, request_usage: {requests}, text_idx: {row.name}, text_id: {row['id']}\n"
        log_file.write(log_entry)

    print("Progress saved successfully.")
    return


def save_csv(i, ids: list, texts: list, annotations: list, row=None, type=None):
    df = pd.DataFrame({'id': ids, 'text': texts, 'annotation': annotations})
    # Check saved_folder existence
    saved_folder = os.path.join(os.getcwd(), output_path)
    if not os.path.exists(os.path.join(os.getcwd(), saved_folder)):
        os.makedirs(os.path.join(os.getcwd(), saved_folder))

    # Save
    file_name, ext = os.path.splitext(file_input)
    file_name = f"{file_name}_gemma-3_{i}.csv"
    if type == 'backup':
        df.to_csv(os.path.join(saved_folder, f"backup_{file_input}"), encoding='utf-8-sig', index=False)
        print(f"Backup to {'backup_' + file_input}")
    else:
        df.to_csv(os.path.join(saved_folder, f"{file_name}"), encoding='utf-8-sig', index=False)
        print(f"Annotation completed and save to {file_name}")

    if row is not None:
        save_progress(row)


In [20]:
"""
Save: write directly output to csv. List, dataframe is for progress backup
"""

ids, texts, annotations = [], [], []

# for i in range(start_point, stop_point + 1):
for i in range(start_point, start_point + 10):
# for i in range(50, stop_point + 1):
    row = df.loc[i]
    id = row['id']
    tweet = row['text']
    cleaned_text = clean_text(tweet)
    if not should_process_tweet(cleaned_text):
        print(f"Tweet blocked due to safety concerns: {cleaned_text}")
        continue

    contents = prompts[0] + f"\n\n{cleaned_text}"

    response = generate_response(contents)
    if response == -1:
        break

    tokens += response.usage_metadata.total_token_count
    requests += 1

    # Extract response content
    for part in response.parts:
        print(i, id, part.text, sep='\t')
        annotation_parts = part.text.split('\n')
        try:
            for annotation_part in annotation_parts:
                if "**Label:**" in annotation_part:
                    label_part = annotation_part
                    break
            else:
                label_part = annotation_parts[0]

            label = label_part.replace("**Label**:", "").strip()
        except:
            label = "Unable to determine"


    ids.append(id)
    texts.append(tweet)
    annotations.append(label)

    if i % 100 == 99: # Backup
        save_csv(i, ids, texts, annotations, row, type='backup')
        sleep_time = random.randint(15, 30)
        print(f"Play around and sleep for {sleep_time} seconds...")
        time.sleep(10)
        response = generate_response()
        tokens += response.usage_metadata.total_token_count
        requests += 1
        time.sleep(sleep_time - 10)

    # Early break
    if stop_point is not None and i == stop_point:
        break
    if requests > RPD * 0.95:
        break

save_csv(i, ids, texts, annotations, row)

12066	tik010301	**Label**: ["sadness", "grief", "caring", "remorse"]

Rate limiting: Sleeping for 2.00 seconds...
12067	tik010302	**Label**: ["sadness", "grief", "love", "gratitude", "remorse"]

Rate limiting: Sleeping for 2.00 seconds...
12068	tik010303	**Label**: ["love", "sadness", "gratitude"]

Rate limiting: Sleeping for 2.00 seconds...
12069	tik010305	**Label**: ["sadness", "grief"]

Rate limiting: Sleeping for 2.00 seconds...
12070	tik010306	**Label**: ["sadness", "grief"]

Rate limiting: Sleeping for 2.00 seconds...
12071	tik010307	**Label**: ["sadness", "grief", "disappointment"]

Rate limiting: Sleeping for 2.00 seconds...
12072	tik010309	**Label**: ["confusion", "sadness"]

Rate limiting: Sleeping for 2.00 seconds...
12073	tik010310	**Label**: ["sadness", "grief", "remorse"]

Rate limiting: Sleeping for 2.00 seconds...
12074	tik010311	**Label**: ["sadness", "grief", "remorse"]

Rate limiting: Sleeping for 2.00 seconds...
12075	tik010312	**Label**: ["love", "caring"]

Annotat

In [21]:
# In case stop the previous run suddenly, we need to re-map i and row before saving
for i, row in df.iterrows():
    if row['id'] == ids[-1]:
        print(row)
        break

save_csv(i, ids, texts, annotations, row)

Unnamed: 0              12075
id                  tik010312
text          🥰🥰🥰thương ngoại
Name: 12075, dtype: object
Annotation completed and save to comment_25.03.16_gemma-3_12075.csv
Progress saved successfully.


In [19]:
key_status

{}